## Imports

In [1]:
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR


## Dataset import

We must use the unprocessed data, because the preprocessing done before is not useful for this regression task


In [2]:
df_test = pd.read_csv('/home/tsalvador/Documents/Polito/ML4N-PoliTo/Datasets/https_test.csv')
df_train = pd.read_csv('/home/tsalvador/Documents/Polito/ML4N-PoliTo/Datasets/https_training.csv')


for i in df_train.columns: #reromve non numeric columns
    if not(i.startswith('_')):
        df_train.drop(i, axis=1, inplace=True) 


for i in df_test.columns: #reromve non numeric columns
    if not(i.startswith('_')):
        df_test.drop(i, axis=1, inplace=True) 


In [3]:
def regression_metrics(y_true, y_pred, name):
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }


Data preparation (s_bytes_all)

In [4]:

# Target: s_bytes_all
ytr_bytes = df_train["_s_bytes_all"].copy()
yte_bytes = df_test["_s_bytes_all"].copy()

# Remove forbidden column
Xtr_bytes = df_train.drop(columns=["_s_bytes_all", "_s_bytes_uniq"])
Xte_bytes = df_test.drop(columns=["_s_bytes_all", "_s_bytes_uniq"])


Data preparation (s_rtt_avg)

In [5]:
# Remove zero RTT values ONLY ON TRAINING SET
dft = df_train[df_train["_s_rtt_avg"] != 0].copy()

ytr_rtt = dft["_s_rtt_avg"]
yte_rtt = df_test["_s_rtt_avg"]


# Remove all RTT-related columns
Xtr_rtt = dft.drop(
    columns=[c for c in dft.columns if "rtt" in c.lower()]
)

Xte_rtt = df_test.drop(
    columns=[c for c in df_test.columns if "rtt" in c.lower()]
)


Cell 5 — Define models

In [6]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor()
}


Cell 6 — Train & evaluate (default hyperparameters)

In [7]:


tasks = {
    'bytes': {
        'X_train': Xtr_bytes,
        'y_train': ytr_bytes,
        'X_test': Xte_bytes,
        'y_test': yte_bytes,
    },
    'rtt': {
        'X_train': Xtr_rtt,
        'y_train': ytr_rtt,
        'X_test': Xte_rtt,
        'y_test': yte_rtt,
    },
}

results_default = []

for task_name, data in tasks.items():
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

    for name, model in models.items():
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])

        pipe.fit(X_train, y_train)
        results_default.append(
            regression_metrics(y_train, pipe.predict(X_train), f"{name}_{task_name}_train")
        )
        results_default.append(
            regression_metrics(y_test, pipe.predict(X_test), f"{name}_{task_name}_test")
        )

results_default = pd.DataFrame(results_default)
results_default



,model,MAE,RMSE,R2
0,LinearRegression_bytes_train,7359.401018,1.480566e+09,0.999908
1,LinearRegression_bytes_test,8027.511315,2.342223e+09,0.999973
2,RandomForest_bytes_train,2781.527457,7.299906e+10,0.995480
3,RandomForest_bytes_test,35987.627148,5.120481e+13,0.407872
4,HistGradientBoostingRegressor_bytes_train,33530.048466,1.195970e+12,0.925944
5,HistGradientBoostingRegressor_bytes_test,105981.073030,6.673878e+13,0.228238
6,LinearRegression_rtt_train,887.402493,8.446817e+06,0.094038
7,LinearRegression_rtt_test,1059.485307,9.709253e+06,0.035826
8,RandomForest_rtt_train,117.599531,4.226522e+05,0.954669
9,RandomForest_rtt_test,467.064096,4.645779e+06,0.538653


Cell 7 — Hyperparameter grids

In [8]:


param_grids = {
    'LinearRegression': {},

    'RandomForest': {
        # keep the search small to avoid getting stuck on the huge bytes task
        'model__n_estimators': [120, 200],
        'model__max_depth': [None, 20],
        'model__min_samples_leaf': [1, 4],
        'model__max_features': ['sqrt'],
        # subsample per tree to speed up training on 130k+ rows
        'model__max_samples': [0.4, 0.7]
    },

    'HistGradientBoostingRegressor': {
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [None, 8, 16],
        'model__max_iter': [200, 400]
    }
}



Cell 8 — Hyperparameter tuning (cross-validation)

In [9]:


results_tuned = []
best_params = {}

for task_name, data in tasks.items():
    X_train = data['X_train']
    y_train = data['y_train']
    X_test = data['X_test']
    y_test = data['y_test']

    for name, model in models.items():
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('model', model)
        ])

        
        cv_folds = 5
        grid = GridSearchCV(
            estimator=pipe,
            param_grid=param_grids[name],
            scoring='neg_mean_absolute_error',
            cv=cv_folds,
            n_jobs=-1,
            verbose=2
        )

        X_fit, y_fit = X_train, y_train
        # train RF on a subset to avoid extremely long runtimes on bytes
        if name == 'RandomForest' and len(y_train) > 70000:
            X_fit = X_train.sample(n=70000, random_state=42)
            y_fit = y_train.loc[X_fit.index]

        grid.fit(X_fit, y_fit)
        best_params[(task_name, name)] = grid.best_params_
        best = grid.best_estimator_

        results_tuned.append(
            regression_metrics(y_train, best.predict(X_train), f"{name}_{task_name}_train_tuned")
        )
        results_tuned.append(
            regression_metrics(y_test, best.predict(X_test), f"{name}_{task_name}_test_tuned")
        )

results_tuned = pd.DataFrame(results_tuned)
results_tuned



Fitting 5 folds for each of 1 candidates, totalling 5 fits
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.3s
[CV] END .................................................... total time=   3.2s
Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=4, model__n_estimators=120; total time=  44.0s
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=4, model__n_estimators=120; total time=  46.5s
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=4, model__n_estimators=120; total time=  45.8s
[CV] END model__

/home/tsalvador/Documents/Polito/ML4N-PoliTo/myenv-env/lib64/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END model__max_depth=20, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=1, model__n_estimators=200; total time= 1.3min
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.7, model__min_samples_leaf=4, model__n_estimators=200; total time= 2.1min
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.7, model__min_samples_leaf=4, model__n_estimators=200; total time= 2.1min
[CV] END model__max_depth=20, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=1, model__n_estimators=200; total time= 1.3min
[CV] END model__max_depth=20, model__max_features=sqrt, model__max_samples=0.4, model__min_samples_leaf=4, model__n_estimators=120; total time=  44.7s
[CV] END model__max_depth=None, model__max_features=sqrt, model__max_samples=0.7, model__min_samples_leaf=4, model__n_estimators=200; total time= 2.3min
[CV] END model__max_depth=20, model__max_features=sqrt, model__max_samples=0.4, model__m

KeyboardInterrupt: 

Cell 9 — Final MAE comparison   

In [ ]:


summary = results_tuned.copy()
summary['target'] = summary['model'].apply(lambda m: 's_bytes_all' if 'bytes' in m else 's_rtt_avg')
summary['split'] = summary['model'].apply(lambda m: 'train' if '_train' in m else 'test')

print('Best hyperparameters from CV:')
for (target, model), params in best_params.items():
    print(f"  {target} - {model}: {params}")

print('Tuned results:')
display(summary[['model', 'target', 'split', 'MAE', 'RMSE', 'R2']])

best_mae = summary[summary['split'] == 'test'].groupby('target')['MAE'].min()
print('Best MAE by target (test set):')
print(best_mae)
print('Units: MAE for s_bytes_all is in bytes; MAE for s_rtt_avg is in milliseconds.')


